In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q transformers peft datasets accelerate

In [ ]:
!pip install -q "peft==0.15.2"

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,DefaultDataCollator
)

from datasets import Dataset
import torch
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

In [ ]:
#Question-1:

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print("Answer:", train.loc[150, "answer"])
print("Encoded label:", train.loc[150, "label"])

In [ ]:
#Question-2:

row0 = train.iloc[0]

option_b_input = str(row0["prompt"]) + " [SEP] " + str(row0["B"])

print(option_b_input)
print("Character length:", len(option_b_input))

In [ ]:
#Question-3:

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.iloc[0]

choices = []

for option in ["A", "B", "C", "D", "E"]:
    choices.append(
        str(row["prompt"]) + " [SEP] " + str(row[option])
    )

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)

print(input_ids.shape)

In [ ]:
#Question-4:

batch_size = 16
num_choices = 5
sequence_length = 128

print(batch_size * num_choices * sequence_length)

In [ ]:
#Question-5

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0)
}

outputs = mc_model(**inputs)

print(outputs.logits.shape)

In [ ]:
#Question-6

row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0),
    "labels": torch.tensor([row["label"]])
}

outputs = mc_model(**inputs)

print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)
print("Number of dimensions:", outputs.loss.dim())

In [ ]:
#Question-7

# Reload a fresh multiple-choice model
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA
mc_model = get_peft_model(mc_model, lora_config)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in mc_model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)

In [ ]:
import transformers, peft, torch
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)

In [ ]:
train_small = train.iloc[:32].copy()

dataset = Dataset.from_pandas(train_small)

In [ ]:
def preprocess(example):
    choices = [
        str(example["prompt"]) + " [SEP] " + str(example[option])
        for option in ["A", "B", "C", "D", "E"]
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    encoding["labels"] = example["label"]
    return encoding

tokenized_dataset = dataset.map(preprocess)

In [ ]:
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=mc_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DefaultDataCollator(),
)

In [ ]:
#Question-9
train_result = trainer.train()

print("Global Step:", trainer.state.global_step)

In [ ]:
#Question-10

mc_model.eval()

device = next(mc_model.parameters()).device

row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0).to(device)
attention_mask = encoding["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = mc_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probs = torch.softmax(outputs.logits, dim=1)

print("Probabilities:", probs.cpu())
print("Option E probability:", probs[0, 4].item())

In [ ]:
#Output:
#q1.